# CS1-EXP1-RF — SVD + Static Features + Random Forest


## 1. Mount Google Drive


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 2. Define shared paths


In [2]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData"
)

RAW_DIR = DRIVE_ROOT / "raw"
PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"

EXPERIMENT_ID = "cs1_project_holdout20_innercv_v1"

EXPERIMENT_MANIFEST_DIR = MANIFEST_ROOT / EXPERIMENT_ID
EXPERIMENT_OUTPUT_DIR = OUTPUT_ROOT / EXPERIMENT_ID

OUTER_SPLIT_DIR = EXPERIMENT_MANIFEST_DIR / "outer_holdout"
INNER_SPLIT_DIR = EXPERIMENT_MANIFEST_DIR / "inner_cv"

EXP1_INNER_CV_OUTPUT_DIR = EXPERIMENT_OUTPUT_DIR / "exp1_inner_cv"
EXP1_FINAL_HOLDOUT_OUTPUT_DIR = EXPERIMENT_OUTPUT_DIR / "exp1_final_holdout"

STATIC_FEATURE_DIR = PROCESSED_DIR / "static_features"

for directory in [
    RAW_DIR,
    PROCESSED_DIR,
    MANIFEST_ROOT,
    OUTPUT_ROOT,
    EXPERIMENT_MANIFEST_DIR,
    EXPERIMENT_OUTPUT_DIR,
    OUTER_SPLIT_DIR,
    INNER_SPLIT_DIR,
    EXP1_INNER_CV_OUTPUT_DIR,
    EXP1_FINAL_HOLDOUT_OUTPUT_DIR,
    STATIC_FEATURE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

NORMALIZED_DATA_PATH = (
    PROCESSED_DIR / "rdiversevul_cs1_normalized_v1.parquet"
)

OUTER_MANIFEST_PATH = (
    OUTER_SPLIT_DIR / "cs1_outer_project_holdout_manifest.parquet"
)

INNER_MANIFEST_PATH = (
    INNER_SPLIT_DIR / "cs1_project_grouped_5fold_manifest.parquet"
)

INNER_SELECTION_METADATA_PATH = (
    INNER_SPLIT_DIR / "cs1_inner_grouped_split_selection_metadata.json"
)

STATIC_FEATURE_PATH = (
    STATIC_FEATURE_DIR / "cs1_static_features_v1.parquet"
)

STATIC_FEATURE_SUMMARY_PATH = (
    STATIC_FEATURE_DIR / "cs1_static_features_v1_summary.csv"
)

STATIC_FEATURE_METADATA_PATH = (
    STATIC_FEATURE_DIR / "cs1_static_features_v1_metadata.json"
)

print("Normalized data:", NORMALIZED_DATA_PATH)
print("Frozen outer manifest:", OUTER_MANIFEST_PATH)
print("Frozen inner manifest:", INNER_MANIFEST_PATH)
print("Static-feature cache:", STATIC_FEATURE_PATH)
print("EXP-1 inner-CV output:", EXP1_INNER_CV_OUTPUT_DIR)


Normalized data: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_v1.parquet
Frozen outer manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_project_holdout_manifest.parquet
Frozen inner manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet
Static-feature cache: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/static_features/cs1_static_features_v1.parquet
EXP-1 inner-CV output: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_project_holdout20_innercv_v1/exp1_inner_cv


## 3. Clone or refresh the `prashant` branch


In [3]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
BRANCH = "prashant"
REPO_DIR = Path("/content/DiverseVul--IS-Project")
PROJECT_DIR = REPO_DIR / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}

if not SRC_DIR.exists():
    raise FileNotFoundError(
        f"Expected source directory does not exist: {SRC_DIR}"
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_DIR)
print("Project:", PROJECT_DIR)
print("Python source path:", SRC_DIR)


Cloning into '/content/DiverseVul--IS-Project'...
remote: Enumerating objects: 237, done.
remote: Counting objects: 100% (237/237), done.
remote: Compressing objects: 100% (171/171), done.
remote: Total 237 (delta 138), reused 161 (delta 62), pack-reused 0 (from 0)
Receiving objects: 100% (237/237), 1.72 MiB | 22.83 MiB/s, done.
Resolving deltas: 100% (138/138), done.
Repository: /content/DiverseVul--IS-Project
Project: /content/DiverseVul--IS-Project/vuln-detection
Python source path: /content/DiverseVul--IS-Project/vuln-detection/src


## 4. Required repository files


In [ ]:
required_repo_files = [
    SRC_DIR / "case_study_1" / "exp1" / "__init__.py",
    SRC_DIR / "case_study_1" / "exp1" / "exp1_rf.py",
    SRC_DIR / "case_study_1" / "exp1" / "static_features.py",
]

missing_repo_files = [
    str(path)
    for path in required_repo_files
    if not path.exists()
]

if missing_repo_files:
    raise FileNotFoundError(
        "EXP-1 repository files are missing:\n"
        + "\n".join(missing_repo_files)
    )

print("EXP-1 repository files are present.")


✅ EXP-1 repository files are present.


## 5. Install dependencies


In [5]:
!pip -q install numpy pandas scipy scikit-learn matplotlib pyyaml pyarrow joblib


## 6. Import the exact EXP-1 modules


In [ ]:
import importlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

exp1_rf = importlib.import_module("case_study_1.exp1.exp1_rf")
static_features = importlib.import_module("case_study_1.exp1.static_features")
split_manifest = importlib.import_module("case_study_1.split_manifest")
evaluation = importlib.import_module("case_study_1.evaluation")

required_exp1_api = [
    "EXP1_VERSION",
    "Exp1Config",
    "run_exp1_profile_fold",
    "run_exp1",
    "run_exp1_final_holdout",
]

missing_exp1_api = [
    name
    for name in required_exp1_api
    if not hasattr(exp1_rf, name)
]

if missing_exp1_api:
    raise AttributeError(
        "EXP-1 runner is missing required API: "
        f"{missing_exp1_api}"
    )

print("EXP-1 runner:", exp1_rf.__file__)
print("Version:", exp1_rf.EXP1_VERSION)
print("Static features:", static_features.__file__)
print("Version:", static_features.STATIC_FEATURE_VERSION)
print("Feature count:", len(static_features.FEATURE_COLUMNS))
print("Split manifest:", split_manifest.__file__)
print("Evaluation:", evaluation.__file__)


✅ EXP-1 runner: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/exp1/exp1_rf.py
   Version: cs1-exp1-svd-static-rf-v3-holdout-innercv-oob
✅ Static features: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/exp1/static_features.py
   Version: cs1-static-features-v1
   Feature count: 54
✅ Split manifest: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/split_manifest.py
✅ Evaluation: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/evaluation.py


## 7. Load the frozen normalized data, outer split, and inner development manifest


In [7]:
if not NORMALIZED_DATA_PATH.exists():
    raise FileNotFoundError(
        "Normalized dataset cache is missing. Run the EXP-0 setup notebook "
        "through normalization first."
    )

if not OUTER_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "Frozen outer holdout manifest is missing. Do not recreate it here."
    )

if not INNER_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "Frozen inner development manifest is missing. Do not recreate it here."
    )

normalized_df = pd.read_parquet(NORMALIZED_DATA_PATH)
outer_manifest_df = pd.read_parquet(OUTER_MANIFEST_PATH)
inner_manifest_df = pd.read_parquet(INNER_MANIFEST_PATH)

required_normalized_columns = {
    "source_row_id",
    "code",
    "normalized_code",
    "label",
    "project",
}

missing_normalized_columns = (
    required_normalized_columns - set(normalized_df.columns)
)

if missing_normalized_columns:
    raise KeyError(
        "Normalized cache is missing required columns: "
        f"{sorted(missing_normalized_columns)}"
    )

required_outer_columns = {
    "source_row_id",
    "label",
    "project",
    "partition",
    "outer_holdout_fold",
}

missing_outer_columns = required_outer_columns - set(outer_manifest_df.columns)

if missing_outer_columns:
    raise KeyError(
        "Outer manifest is missing required columns: "
        f"{sorted(missing_outer_columns)}"
    )

if outer_manifest_df["source_row_id"].duplicated().any():
    raise ValueError("Outer manifest contains duplicate source_row_id values.")

if set(outer_manifest_df["source_row_id"]) != set(normalized_df["source_row_id"]):
    raise ValueError(
        "Outer manifest source IDs do not exactly match normalized data."
    )

if set(outer_manifest_df["partition"].unique()) != {
    "development",
    "outer_holdout",
}:
    raise ValueError(
        "Outer manifest must contain development and outer_holdout partitions."
    )

partition_lookup = outer_manifest_df[
    [
        "source_row_id",
        "partition",
    ]
]

partitioned_df = normalized_df.merge(
    partition_lookup,
    on="source_row_id",
    how="left",
    validate="one_to_one",
)

if partitioned_df["partition"].isna().any():
    raise ValueError("Some normalized rows have no outer partition assignment.")

dev_df = (
    partitioned_df.loc[
        partitioned_df["partition"] == "development"
    ]
    .drop(columns="partition")
    .reset_index(drop=True)
)

holdout_df = (
    partitioned_df.loc[
        partitioned_df["partition"] == "outer_holdout"
    ]
    .drop(columns="partition")
    .reset_index(drop=True)
)

selected_inner_split_seed = None
if INNER_SELECTION_METADATA_PATH.exists():
    with INNER_SELECTION_METADATA_PATH.open("r", encoding="utf-8") as file:
        inner_selection_metadata = json.load(file)
    selected_inner_split_seed = inner_selection_metadata.get(
        "selected_split_seed"
    )

INNER_N_SPLITS = 5

selected_inner_split_config = split_manifest.SplitConfig(
    n_splits=INNER_N_SPLITS,
    random_state=(
        int(selected_inner_split_seed)
        if selected_inner_split_seed is not None
        else 42
    ),
    shuffle=True,
)

split_manifest.assert_manifest_integrity(
    inner_manifest_df,
    config=selected_inner_split_config,
)

print("Normalized rows:", f"{len(normalized_df):,}")
print("Development rows:", f"{len(dev_df):,}")
print("Outer holdout rows:", f"{len(holdout_df):,}")
print("Selected inner split seed:", selected_inner_split_config.random_state)


Normalized rows: 261,667
Development rows: 203,958
Outer holdout rows: 57,709
Selected inner split seed: 674


## 8. Confirm split reuse and project isolation


In [ ]:
dev_ids = set(dev_df["source_row_id"])
holdout_ids = set(holdout_df["source_row_id"])
inner_ids = set(inner_manifest_df["source_row_id"])

assert dev_ids.isdisjoint(holdout_ids)
assert inner_ids == dev_ids
assert inner_ids.isdisjoint(holdout_ids)
assert len(inner_manifest_df) == len(dev_df)
assert inner_manifest_df["source_row_id"].nunique() == len(dev_df)

dev_projects = set(dev_df["project"].astype(str))
holdout_projects = set(holdout_df["project"].astype(str))

assert dev_projects.isdisjoint(holdout_projects)

inner_fold_summary = split_manifest.summarize_manifest(
    inner_manifest_df,
    config=selected_inner_split_config,
)

assert (
    inner_fold_summary["train_test_project_overlap"] == 0
).all()

assert (inner_fold_summary["test_vulnerable"] > 0).all()
assert (inner_fold_summary["test_non_vulnerable"] > 0).all()

print("EXP-1 will reuse exactly the EXP-0 development rows.")
print("Outer holdout rows in inner CV: 0")
print("Project overlap between dev and outer holdout: 0")
print("Project overlap inside every inner fold: 0")

display(
    inner_fold_summary.style.format(
        {
            "test_positive_rate": "{:.4%}",
            "positive_rate_delta_from_global": "{:+.4%}",
            "test_row_share": "{:.2%}",
            "test_project_share": "{:.2%}",
        }
    )
)


✅ EXP-1 will reuse exactly the EXP-0 development rows.
✅ Outer holdout rows in inner CV: 0
✅ Project overlap between dev and outer holdout: 0
✅ Project overlap inside every inner fold: 0


,fold,test_rows,test_vulnerable,test_non_vulnerable,test_positive_rate,positive_rate_delta_from_global,test_unique_projects,train_rows,train_unique_projects,train_test_project_overlap,test_row_share,test_project_share
0,0,55509,2420,53089,4.3597%,-0.8998%,1,148449,593,0,27.22%,0.17%
1,1,40791,2091,38700,5.1261%,-0.1333%,158,163167,436,0,20.00%,26.60%
2,2,39072,2062,37010,5.2774%,+0.0180%,146,164886,448,0,19.16%,24.58%
3,3,35016,2047,32969,5.8459%,+0.5865%,137,168942,457,0,17.17%,23.06%
4,4,33570,2107,31463,6.2764%,+1.0170%,152,170388,442,0,16.46%,25.59%


## 9. Load or create the deterministic static-feature cache

The cache uses only `source_row_id` and raw `code`; it does not use labels,
project identifiers, TF-IDF corpus statistics, scaling, or trained model
parameters. Building it globally is therefore acceptable. During EXP-1
model fitting, only development rows are supplied to the runner.


In [ ]:
if STATIC_FEATURE_PATH.exists():
    static_df = pd.read_parquet(STATIC_FEATURE_PATH)
    print("Loaded existing static-feature cache.")
else:
    print("Static-feature cache not found. Starting one-time extraction...")

    static_config = static_features.StaticFeatureConfig(
        source_id_column="source_row_id",
        code_column="code",
        progress_every=25_000,
    )

    static_df = static_features.extract_static_feature_frame(
        normalized_df[
            [
                "source_row_id",
                "code",
            ]
        ],
        config=static_config,
    )

    static_artifacts = static_features.save_static_feature_artifacts(
        static_frame=static_df,
        output_dir=STATIC_FEATURE_DIR,
        config=static_config,
        source_dataset_path=NORMALIZED_DATA_PATH,
    )

    print("Static features extracted and saved.")
    for artifact_name, artifact_path in static_artifacts.items():
        print(f"  {artifact_name}: {artifact_path}")

print("Static feature table shape:", static_df.shape)
display(static_df.head())


✅ Loaded existing static-feature cache.
Static feature table shape: (261667, 55)


,source_row_id,raw_char_count,raw_line_count,nonempty_line_count,avg_nonempty_line_length,max_line_length,comment_char_count,comment_line_count,comment_char_ratio,string_literal_count,...,memory_api_call_count,string_api_call_count,format_api_call_count,input_api_call_count,allocation_api_call_count,deallocation_api_call_count,unsafe_api_presence_count,sizeof_count,null_token_count,assert_call_count
0,0,6098.0,105.0,93.0,64.451613,151.0,52.0,1.0,0.008527,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,1732.0,65.0,55.0,30.327273,71.0,185.0,2.0,0.106813,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,115.0,4.0,4.0,28.000000,59.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,289.0,12.0,10.0,27.800000,50.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,824.0,27.0,23.0,34.695652,78.0,0.0,0.0,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 10. Validate the static-feature cache


In [ ]:
feature_columns = static_features.FEATURE_COLUMNS

assert len(feature_columns) == 54
assert len(static_df) == len(normalized_df)
assert static_df["source_row_id"].nunique() == len(normalized_df)
assert set(static_df["source_row_id"]) == set(normalized_df["source_row_id"])

expected_static_columns = [
    "source_row_id",
    *feature_columns,
]

if list(static_df.columns) != expected_static_columns:
    raise ValueError(
        "Static feature schema mismatch.\n"
        f"Expected: {expected_static_columns}\n"
        f"Actual:   {list(static_df.columns)}"
    )

static_values = static_df[feature_columns].to_numpy(dtype=np.float32)

assert np.isfinite(static_values).all()
assert (static_values >= 0).all()

assert set(dev_df["source_row_id"]).issubset(
    set(static_df["source_row_id"])
)
assert set(holdout_df["source_row_id"]).issubset(
    set(static_df["source_row_id"])
)

print("Static feature cache is complete and numerically valid.")
print("Feature count:", len(feature_columns))
print("Cache scope: full normalized dataset (label-free deterministic features)")


✅ Static feature cache is complete and numerically valid.
Feature count: 54
Cache scope: full normalized dataset (label-free deterministic features)


## 11. Smoke-test the static extractor


In [11]:
sample_code = """
int copy_payload(char *dst, const char *src, size_t n) {
    char buffer[64];

    if (n < sizeof(buffer)) {
        memcpy(buffer, src, n);
        buffer[n] = '\\0';
    }

    return 0;
}
"""

sample_features = static_features.extract_static_features(sample_code)

selected_sample_features = {
    name: sample_features[name]
    for name in [
        "if_count",
        "sizeof_count",
        "memory_api_call_count",
        "pointer_declaration_count",
        "array_access_count",
        "cyclomatic_complexity_proxy",
    ]
}

print(selected_sample_features)

assert sample_features["if_count"] == 1
assert sample_features["sizeof_count"] == 1
assert sample_features["memory_api_call_count"] == 1
assert sample_features["pointer_declaration_count"] >= 1
assert sample_features["array_access_count"] >= 1


{'if_count': 1.0, 'sizeof_count': 1.0, 'memory_api_call_count': 1.0, 'pointer_declaration_count': 2.0, 'array_access_count': 2.0, 'cyclomatic_complexity_proxy': 2.0}


## 12. Declare the frozen EXP-1 configuration


In [ ]:
INNER_MODEL_RANDOM_STATE = 42

exp1_config = exp1_rf.Exp1Config(
    experiment_name="cs1_exp1_rf_inner_dev_grouped",
    code_column="abstracted_code_v1",
    source_id_column="source_row_id",
    label_column="label",
    project_column="project",
    fold_column="fold",
    n_splits=INNER_N_SPLITS,
    random_state=INNER_MODEL_RANDOM_STATE,
    decision_threshold=0.50,
    word_ngram_range=(1, 3),
    word_min_df=3,
    word_max_df=0.995,
    word_max_features=50_000,
    char_analyzer="char",
    char_ngram_range=(3, 4),
    char_min_df=8,
    char_max_df=0.995,
    char_max_features=60_000,
    svd_n_components=256,
    svd_algorithm="randomized",
    svd_n_iter=5,
    svd_n_oversamples=10,
    rf_n_estimators=200,
    rf_criterion="gini",
    rf_max_depth=None,
    rf_min_samples_split=2,
    rf_min_samples_leaf=2,
    rf_max_features="sqrt",
    rf_bootstrap=True,
    rf_max_samples=None,
    rf_class_weight="balanced_subsample",
    rf_n_jobs=-1,
    feature_importance_top_n=50,
    verbose=True,
)

print(exp1_config)
print("\nSplit seed:", selected_inner_split_config.random_state)
print("Model/SVD random state:", INNER_MODEL_RANDOM_STATE)
print("Final outer holdout: locked and untouched")

Exp1Config(experiment_name='cs1_exp1_rf_inner_dev_grouped', code_column='normalized_code', source_id_column='source_row_id', label_column='label', project_column='project', fold_column='fold', n_splits=5, random_state=42, decision_threshold=0.5, word_ngram_range=(1, 3), word_min_df=3, word_max_df=0.995, word_max_features=50000, char_analyzer='char', char_ngram_range=(3, 4), char_min_df=8, char_max_df=0.995, char_max_features=60000, lowercase=False, sublinear_tf=True, tfidf_norm='l2', svd_n_components=256, svd_algorithm='randomized', svd_n_iter=5, svd_n_oversamples=10, rf_n_estimators=200, rf_criterion='gini', rf_max_depth=None, rf_min_samples_split=2, rf_min_samples_leaf=2, rf_max_features='sqrt', rf_bootstrap=True, rf_max_samples=None, rf_class_weight='balanced_subsample', rf_n_jobs=-1, rf_oob_score=True, oob_threshold_min=0.005, oob_threshold_max=0.25, oob_threshold_step=0.005, oob_threshold_objective='f1', feature_importance_top_n=50, verbose=True)

Split seed: 674
Model/SVD random 

In [ ]:
import pandas as pd

dev_df['abstracted_code_v1'] = dev_df['abstracted_code_v1'].fillna("").astype(str)
empty_dev_mask = dev_df['abstracted_code_v1'].str.strip().eq("")
if empty_dev_mask.any():
    dev_df.loc[empty_dev_mask, 'abstracted_code_v1'] = "EMPTY_ABSTRACTED_CODE_SAMPLE"

holdout_df['abstracted_code_v1'] = holdout_df['abstracted_code_v1'].fillna("").astype(str)
empty_holdout_mask = holdout_df['abstracted_code_v1'].str.strip().eq("")
if empty_holdout_mask.any():
    holdout_df.loc[empty_holdout_mask, 'abstracted_code_v1'] = "EMPTY_ABSTRACTED_CODE_SAMPLE"

print("Preprocessing guards completed for development and holdout frames.")

## 13. Profile the most demanding inner fold

This profile uses the frozen development manifest only. It runs the fold
with the largest training partition so that SVD, memory use, and Random
Forest training are checked before official five-fold execution.


In [13]:
PROFILE_FOLD_ID = int(
    inner_fold_summary.loc[
        inner_fold_summary["train_rows"].idxmax(),
        "fold",
    ]
)

profile_train_rows = int(
    inner_fold_summary.loc[
        inner_fold_summary["fold"] == PROFILE_FOLD_ID,
        "train_rows",
    ].iloc[0]
)

print(
    "Profiling the largest training partition: "
    f"Fold {PROFILE_FOLD_ID} with {profile_train_rows:,} training rows."
)

exp1_profile_results = exp1_rf.run_exp1_profile_fold(
    normalized_frame=dev_df,
    static_features_frame=static_df,
    manifest=inner_manifest_df,
    fold_id=PROFILE_FOLD_ID,
    config=exp1_config,
)


Profiling the largest training partition: Fold 4 with 170,388 training rows.
[10:17:40] CS1-EXP1 profiling mode: running Fold 5/5 only.
[10:17:42] Fold 5/5 started | train=170,388, test=33,570, train projects=442, test projects=152.
[10:17:42] Fold 5/5 | fitting word TF-IDF...
[10:19:37] Fold 5/5 | word TF-IDF done in 1.92 min (50,000 features).
[10:19:37] Fold 5/5 | fitting character TF-IDF...
[10:23:11] Fold 5/5 | character TF-IDF done in 3.56 min (60,000 features).
[10:23:11] Fold 5/5 | joining sparse lexical matrices...
[10:23:12] Fold 5/5 | sparse lexical matrix ready in 1.2s (110,000 columns).
[10:23:12] Fold 5/5 | fitting train-only TruncatedSVD (256 components)...
[10:27:11] Fold 5/5 | TruncatedSVD done in 3.97 min (explained variance ratio sum=0.2143).
[10:27:11] Fold 5/5 | loading cached static features...
[10:27:11] Fold 5/5 | combining SVD and static features...
[10:27:11] Fold 5/5 | training Random Forest (trees=200, max_features=sqrt, class_weight=balanced_subsample)...
[

## 14. Inspect the EXP-1 computational profile


In [ ]:


print("Largest-fold computational profile:")
display(exp1_profile_results["training_metadata"])

profile_predictions_df = exp1_profile_results["predictions"].copy()

has_oob_threshold = (
    "selected_oob_threshold" in profile_predictions_df.columns
)

if has_oob_threshold:
    selected_threshold_values = (
        profile_predictions_df["selected_oob_threshold"]
        .dropna()
        .unique()
    )

    if len(selected_threshold_values) != 1:
        raise RuntimeError(
            "Expected exactly one selected OOB threshold for this profile fold, "
            f"but found: {selected_threshold_values}"
        )

    selected_threshold = float(selected_threshold_values[0])

    print(
        "\nThreshold strategy: training-fold OOB selection "
        "(correct leakage-safe operating point)."
    )
    print(f"Selected OOB threshold: {selected_threshold:.3f}")

else:
    selected_threshold = None

    print(
        "\n⚠️ No 'selected_oob_threshold' column was returned by the current "
        "EXP-1 runner."
    )
    print(
        "The displayed thresholded metrics therefore use the fixed "
        "decision_threshold from Exp1Config, currently 0.50."
    )
    print(
        "For Random Forest, 0.50 may be unsuitable because its score scale "
        "is different from the SGD Logistic Regression score scale."
    )

profile_metrics_df = pd.DataFrame(
    list(exp1_profile_results["profile_metrics"].items()),
    columns=["metric", "value"],
)

print(
    "\nLargest-fold primary descriptive metrics "
    "— not the official five-fold EXP-1 result:"
)
display(profile_metrics_df)

if "default_threshold_metrics" in exp1_profile_results:
    default_threshold_metrics_df = pd.DataFrame(
        list(
            exp1_profile_results[
                "default_threshold_metrics"
            ].items()
        ),
        columns=["metric", "value"],
    )

    print(
        "\nFixed threshold = 0.50 diagnostic only "
        "— not the selected EXP-1 operating point:"
    )
    display(default_threshold_metrics_df)

score_summary_df = pd.DataFrame(
    {
        "statistic": [
            "minimum_score",
            "mean_score",
            "median_score",
            "95th_percentile_score",
            "99th_percentile_score",
            "maximum_score",
            "positive_rate_in_test_fold",
        ],
        "value": [
            float(profile_predictions_df["y_score"].min()),
            float(profile_predictions_df["y_score"].mean()),
            float(profile_predictions_df["y_score"].median()),
            float(profile_predictions_df["y_score"].quantile(0.95)),
            float(profile_predictions_df["y_score"].quantile(0.99)),
            float(profile_predictions_df["y_score"].max()),
            float(profile_predictions_df["label"].mean()),
        ],
    }
)

print("\nRandom Forest held-out score distribution:")
display(score_summary_df)

print("\nTop static-feature impurity importances for this profile fold:")
display(
    exp1_profile_results["feature_importances"]
    .loc[
        lambda frame: frame["feature_group"] == "static"
    ]
    .sort_values("rank")
    .head(20)
)

Largest-fold computational profile:


,fold,train_rows,test_rows,train_vulnerable,test_vulnerable,train_positive_rate,test_positive_rate,train_unique_projects,test_unique_projects,word_features,...,oob_precision,oob_recall,oob_f1,oob_mcc,prediction_seconds,total_fold_seconds,optimizer,score_min,score_max,score_mean
0,4,170388,33570,8620,2107,0.05059,0.062764,442,152,50000,...,0.187885,0.318794,0.236428,0.191966,1.376326,1464.703075,RandomForestClassifier,0.0,0.666274,0.066721



Threshold strategy: training-fold OOB selection (correct leakage-safe operating point).
Selected OOB threshold: 0.150

Largest-fold primary descriptive metrics — not the official five-fold EXP-1 result:


,metric,value
0,n_samples,33570
1,vulnerable_1,2107
2,non_vulnerable_0,31463
3,positive_rate,0.062764
4,threshold,0.15
5,average_precision_pr_auc,0.157396
6,precision,0.191557
7,recall,0.312292
8,f1,0.237459
9,mcc,0.179283



Fixed threshold = 0.50 diagnostic only — not the selected EXP-1 operating point:


,metric,value
0,n_samples,33570.000000
1,vulnerable_1,2107.000000
2,non_vulnerable_0,31463.000000
3,positive_rate,0.062764
4,threshold,0.500000
5,average_precision_pr_auc,0.157396
6,precision,0.157895
7,recall,0.001424
8,f1,0.002822
9,mcc,0.009334



Random Forest held-out score distribution:


,statistic,value
0,minimum_score,0.000000
1,mean_score,0.066721
2,median_score,0.044014
3,95th_percentile_score,0.215360
4,99th_percentile_score,0.346525
5,maximum_score,0.666274
6,positive_rate_in_test_fold,0.062764



Top static-feature impurity importances for this profile fold:


,fold,feature,feature_group,importance,rank
0,4,static::identifier_count,static,0.017508,1
1,4,static::raw_char_count,static,0.014436,2
2,4,static::unique_identifier_count,static,0.014206,3
3,4,static::nonempty_line_count,static,0.013598,4
4,4,static::raw_line_count,static,0.011884,5
5,4,static::assignment_operator_count,static,0.011312,6
6,4,static::function_call_count,static,0.010882,7
7,4,static::parenthesis_open_count,static,0.009820,8
9,4,static::identifier_diversity,static,0.008257,10
10,4,static::brace_open_count,static,0.007795,11


## 15. Run official five-fold development CV

Leave this switch `False` until the profile completed without memory errors,
SVD errors, or unexpected data-integrity failures.


In [15]:
RUN_INNER_CV = False

if RUN_INNER_CV:
    existing_files = list(EXP1_INNER_CV_OUTPUT_DIR.iterdir())

    if existing_files:
        raise RuntimeError(
            "EXP-1 inner-CV output directory is not empty. "
            "Do not mix reruns. Use a new experiment version or remove "
            "only incomplete artifacts deliberately."
        )

    exp1_inner_results = exp1_rf.run_exp1(
        normalized_frame=dev_df,
        static_features_frame=static_df,
        manifest=inner_manifest_df,
        config=exp1_config,
        output_dir=EXP1_INNER_CV_OUTPUT_DIR,
        additional_metadata={
            "evaluation_stage": "inner_development_cv",
            "evaluation_protocol": (
                "Frozen project-disjoint outer holdout; frozen inner "
                "5-fold grouped CV on development projects."
            ),
            "normalized_dataset_path": str(NORMALIZED_DATA_PATH),
            "outer_holdout_manifest_path": str(OUTER_MANIFEST_PATH),
            "inner_manifest_path": str(INNER_MANIFEST_PATH),
            "static_feature_cache_path": str(STATIC_FEATURE_PATH),
            "candidate_selection_rule": (
                "Compare candidates using pooled development OOF PR-AUC. "
                "Outer-holdout labels are excluded from selection."
            ),
            "representation": (
                "Train-only word/character TF-IDF, train-only TruncatedSVD, "
                "and 54 deterministic source-level static proxy features."
            ),
        },
    )

    print(
        evaluation.format_metric_report(
            exp1_inner_results["evaluation"]["pooled_metrics"]
        )
    )
else:
    print(
        "Inner CV is locked. Set RUN_INNER_CV = True only after the "
        "profile is accepted."
    )


[10:51:03] CS1-EXP1 official run started: 5-fold grouped CV.
[10:51:03] Configuration: lexical<= 110,000, SVD=256, static=54, RF trees=200.
[10:51:04] Fold 1/5 started | train=148,449, test=55,509, train projects=593, test projects=1.
[10:51:04] Fold 1/5 | fitting word TF-IDF...
[10:52:58] Fold 1/5 | word TF-IDF done in 1.89 min (50,000 features).
[10:52:58] Fold 1/5 | fitting character TF-IDF...
[10:56:36] Fold 1/5 | character TF-IDF done in 3.63 min (60,000 features).
[10:56:36] Fold 1/5 | joining sparse lexical matrices...
[10:56:37] Fold 1/5 | sparse lexical matrix ready in 1.2s (110,000 columns).
[10:56:37] Fold 1/5 | fitting train-only TruncatedSVD (256 components)...
[11:00:13] Fold 1/5 | TruncatedSVD done in 3.60 min (explained variance ratio sum=0.2151).
[11:00:13] Fold 1/5 | loading cached static features...
[11:00:13] Fold 1/5 | combining SVD and static features...
[11:00:13] Fold 1/5 | training Random Forest (trees=200, max_features=sqrt, class_weight=balanced_subsample)...

## 16. Review the official inner-CV result


In [ ]:
if "exp1_inner_results" not in globals():
    raise RuntimeError(
        "exp1_inner_results is not available in this runtime. "
        "Do not rerun the full CV. Reload saved artifacts from Drive instead."
    )


standard_evaluation = exp1_inner_results["evaluation"]
standard_pooled_metrics = standard_evaluation["pooled_metrics"]

ranking_metric_names = [
    "n_samples",
    "vulnerable_1",
    "non_vulnerable_0",
    "positive_rate",
    "average_precision_pr_auc",
]

ranking_metrics_df = pd.DataFrame(
    [
        {
            "metric": metric_name,
            "value": standard_pooled_metrics.get(metric_name),
        }
        for metric_name in ranking_metric_names
    ]
)

print("EXP-1 pooled development OOF ranking metrics:")
print(
    "These are threshold-independent. "
    "PR-AUC is the primary comparison metric against EXP-0."
)
display(ranking_metrics_df)

oob_operating_evaluation = exp1_inner_results[
    "oob_operating_evaluation"
]

oob_pooled_metrics = oob_operating_evaluation["pooled_metrics"]

oob_metric_names = [
    "n_samples",
    "vulnerable_1",
    "non_vulnerable_0",
    "positive_rate",
    "average_precision_pr_auc",
    "precision",
    "recall",
    "f1",
    "mcc",
    "specificity",
    "false_positive_rate",
    "false_negative_rate",
    "true_negative",
    "false_positive",
    "false_negative",
    "true_positive",
    "predicted_positive",
    "predicted_positive_rate",
    "threshold_strategy",
    "selected_threshold_min",
    "selected_threshold_max",
    "selected_threshold_mean",
]

oob_pooled_metrics_df = pd.DataFrame(
    [
        {
            "metric": metric_name,
            "value": oob_pooled_metrics.get(metric_name),
        }
        for metric_name in oob_metric_names
        if metric_name in oob_pooled_metrics
    ]
)

print(
    "\nEXP-1 pooled OOF operating metrics "
    "(per-fold thresholds selected from training-only OOB predictions):"
)
display(oob_pooled_metrics_df)


print("\nEXP-1 per-fold OOB-threshold metrics:")
display(oob_operating_evaluation["fold_metrics"])

print("\nEXP-1 fold-to-fold operating-metric stability:")
display(oob_operating_evaluation["fold_summary"])


fold_training_df = exp1_inner_results["fold_training"].copy()

threshold_runtime_columns = [
    "fold",
    "train_rows",
    "test_rows",
    "train_unique_projects",
    "test_unique_projects",
    "svd_explained_variance_ratio_sum",
    "selected_oob_threshold",
    "oob_average_precision_pr_auc",
    "oob_precision",
    "oob_recall",
    "oob_f1",
    "oob_mcc",
    "model_fit_seconds",
    "total_fold_seconds",
]

available_columns = [
    column
    for column in threshold_runtime_columns
    if column in fold_training_df.columns
]

print("\nThreshold selection and runtime by fold:")
display(fold_training_df[available_columns])


fixed_threshold_metrics = standard_evaluation["pooled_metrics"]

fixed_threshold_display_names = [
    "threshold",
    "precision",
    "recall",
    "f1",
    "mcc",
    "specificity",
    "false_positive_rate",
    "true_positive",
    "false_positive",
    "false_negative",
    "predicted_positive",
]

fixed_threshold_metrics_df = pd.DataFrame(
    [
        {
            "metric": metric_name,
            "value": fixed_threshold_metrics.get(metric_name),
        }
        for metric_name in fixed_threshold_display_names
    ]
)

print(
    "\nFixed threshold = 0.50 diagnostic only. "
    "Do not use this table as EXP-1's main operating result."
)
display(fixed_threshold_metrics_df)

EXP-1 pooled development OOF ranking metrics:
These are threshold-independent. PR-AUC is the primary comparison metric against EXP-0.


,metric,value
0,n_samples,203958.000000
1,vulnerable_1,10727.000000
2,non_vulnerable_0,193231.000000
3,positive_rate,0.052594
4,average_precision_pr_auc,0.124803



EXP-1 pooled OOF operating metrics (per-fold thresholds selected from training-only OOB predictions):


,metric,value
0,n_samples,203958
1,vulnerable_1,10727
2,non_vulnerable_0,193231
3,positive_rate,0.052594
4,average_precision_pr_auc,0.124803
5,precision,0.153389
6,recall,0.269227
7,f1,0.195432
8,mcc,0.144
9,specificity,0.917508



EXP-1 per-fold OOB-threshold metrics:


...

,metric,mean,std,min,max
0,average_precision_pr_auc,0.126461,0.027101,0.084827,0.157396
1,precision,0.153235,0.027568,0.114143,0.191557
2,recall,0.272930,0.069355,0.160744,0.342452
3,f1,0.195373,0.038778,0.133493,0.237459
4,mcc,0.143500,0.034025,0.088360,0.179283
5,false_positive_rate,0.085618,0.020758,0.056867,0.113774
6,predicted_positive_rate,0.095964,0.024072,0.061395,0.127142
7,threshold,0.137000,0.011511,0.120000,0.150000



Threshold selection and runtime by fold:


,fold,train_rows,test_rows,train_unique_projects,test_unique_projects,svd_explained_variance_ratio_sum,selected_oob_threshold,oob_average_precision_pr_auc,oob_precision,oob_recall,oob_f1,oob_mcc,model_fit_seconds,total_fold_seconds
0,0,148449,55509,593,1,0.215080,0.135,0.165571,0.195584,0.390273,0.260580,0.215376,792.589746,1349.917782
1,1,163167,40791,436,158,0.215080,0.145,0.154509,0.190256,0.343214,0.244807,0.199285,858.055926,1421.175233
2,2,164886,39072,448,146,0.215313,0.135,0.137421,0.170615,0.328332,0.224546,0.177475,850.160839,1382.606400
3,3,168942,35016,457,137,0.210871,0.120,0.147190,0.175569,0.385484,0.241257,0.200592,863.515276,1429.051659
4,4,170388,33570,442,152,0.214253,0.150,0.142995,0.187885,0.318794,0.236428,0.191966,882.312032,1411.208157



Fixed threshold = 0.50 diagnostic only. Do not use this table as EXP-1's main operating result.


,metric,value
0,threshold,0.500000
1,precision,0.207317
2,recall,0.001585
3,f1,0.003146
4,mcc,0.013901
5,specificity,0.999664
6,false_positive_rate,0.000336
7,true_positive,17.000000
8,false_positive,65.000000
9,false_negative,10710.000000


## 17. Inspect stable static-feature importance


In [17]:
if "exp1_inner_results" not in globals():
    print("Run official inner CV first.")
else:
    static_importance_summary = (
        exp1_inner_results["feature_importances"]
        .loc[
            lambda frame: frame["feature_group"] == "static"
        ]
        .groupby("feature", as_index=False)
        .agg(
            mean_importance=("importance", "mean"),
            std_importance=("importance", "std"),
            best_rank=("rank", "min"),
            mean_rank=("rank", "mean"),
        )
        .sort_values(
            ["mean_importance", "best_rank"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    print(
        "Static-feature impurity importance is descriptive only; it is "
        "not a causal explanation."
    )
    display(static_importance_summary.head(25))


Static-feature impurity importance is descriptive only; it is not a causal explanation.


,feature,mean_importance,std_importance,best_rank,mean_rank
0,static::identifier_count,0.018527,0.001103,1,1.0
1,static::raw_char_count,0.015076,0.001486,2,2.2
2,static::unique_identifier_count,0.013946,0.001589,3,3.4
3,static::nonempty_line_count,0.013793,0.000650,2,3.6
4,static::raw_line_count,0.012780,0.001178,4,4.8
5,static::assignment_operator_count,0.010928,0.000654,6,7.0
6,static::function_call_count,0.010732,0.000893,6,7.0
7,static::parenthesis_open_count,0.010215,0.000658,8,8.0
8,static::identifier_diversity,0.009957,0.001295,6,8.2
9,static::brace_open_count,0.007739,0.001374,10,11.6


## 18. Compare EXP-0 and EXP-1 only after both use the same frozen development split


In [ ]:
import pandas as pd

exp0_metrics = {
    "model": "EXP-0-IDABS | TF-IDF + SGD Logistic Regression",
    "pr_auc": 0.125205,
    "precision": 0.131065,
    "recall": 0.377459,
    "f1": 0.194570,
    "mcc": 0.148525,
    "false_positive_rate": 0.138922,
    "true_positive": 4049,
    "false_positive": 26844,
    "predicted_positive": 30893,
    "runtime_minutes": 32.08,
    "operating_policy": "Fixed predeclared threshold = 0.50",
}

exp1_metrics = {
    "model": "EXP-1-IDABS | TF-IDF + SVD + static features + RF",
    "pr_auc": 0.124803,
    "precision": 0.153389,
    "recall": 0.269227,
    "f1": 0.195432,
    "mcc": 0.144000,
    "false_positive_rate": 0.082492,
    "true_positive": 2888,
    "false_positive": 15940,
    "predicted_positive": 18828,
    "runtime_minutes": 116.81,
    "operating_policy": "Per-fold training-only OOB F1 threshold",
}

comparison_df = pd.DataFrame([exp0_metrics, exp1_metrics])
comparison_df["pr_auc_vs_dataset_prevalence"] = comparison_df["pr_auc"] / 0.052594
comparison_df["alerts_per_true_positive"] = comparison_df["predicted_positive"] / comparison_df["true_positive"]
comparison_df["runtime_vs_exp0"] = comparison_df["runtime_minutes"] / comparison_df.loc[0, "runtime_minutes"]

comparison_columns = [
    "model",
    "pr_auc",
    "pr_auc_vs_dataset_prevalence",
    "precision",
    "recall",
    "f1",
    "mcc",
    "false_positive_rate",
    "true_positive",
    "false_positive",
    "predicted_positive",
    "alerts_per_true_positive",
    "runtime_minutes",
    "runtime_vs_exp0",
    "operating_policy",
]

print("Development-only candidate comparison.\nThe frozen outer holdout remains untouched.")
display(comparison_df[comparison_columns].style.format({
    "pr_auc": "{:.6f}",
    "pr_auc_vs_dataset_prevalence": "{:.2f}×",
    "precision": "{:.4f}",
    "recall": "{:.4f}",
    "f1": "{:.4f}",
    "mcc": "{:.4f}",
    "false_positive_rate": "{:.4f}",
    "alerts_per_true_positive": "{:.2f}",
    "runtime_minutes": "{:.2f}",
    "runtime_vs_exp0": "{:.2f}×",
}))

Development-only candidate comparison.
The frozen outer holdout remains untouched.


,model,pr_auc,pr_auc_vs_dataset_prevalence,precision,recall,f1,mcc,false_positive_rate,true_positive,false_positive,predicted_positive,alerts_per_true_positive,runtime_minutes,runtime_vs_exp0,operating_policy
0,EXP-0 | TF-IDF + SGD Logistic Regression,0.125205,2.38×,0.1311,0.3775,0.1946,0.1485,0.1389,4049,26844,30893,7.63,32.08,1.00×,Fixed predeclared threshold = 0.50
1,EXP-1 | TF-IDF + SVD + static features + RF,0.124803,2.37×,0.1534,0.2692,0.1954,0.1440,0.0825,2888,15940,18828,6.52,116.81,3.64×,Per-fold training-only OOB F1 threshold



Development-only selection interpretation:
- Primary ranking metric: EXP-0 has the marginally higher PR-AUC.
- Detection coverage: EXP-0 finds more vulnerable functions.
- Review workload: EXP-1 creates fewer alerts and fewer false positives.
- Practical compute cost: EXP-0 is substantially faster.
- No outer-holdout labels were used in this comparison.


## 19. Final outer holdout remains locked


In [ ]:
FINAL_MODEL_LOCKED = False
FINAL_SELECTED_MODEL = None

if (
    FINAL_MODEL_LOCKED
    and FINAL_SELECTED_MODEL == exp1_config.experiment_name
):
    final_existing_files = list(EXP1_FINAL_HOLDOUT_OUTPUT_DIR.iterdir())

    if final_existing_files:
        raise RuntimeError(
            "Final-holdout output directory is not empty. "
            "The outer holdout must be evaluated once only."
        )

    exp1_final_holdout_results = exp1_rf.run_exp1_final_holdout(
        dev_frame=dev_df,
        holdout_frame=holdout_df,
        static_features_frame=static_df,
        config=exp1_config,
        output_dir=EXP1_FINAL_HOLDOUT_OUTPUT_DIR,
        additional_metadata={
            "evaluation_stage": "single_final_outer_holdout",
            "selected_model_record": FINAL_SELECTED_MODEL,
            "selection_source": (
                "Development-only frozen inner 5-fold grouped CV. "
                "The outer holdout was not used for model, threshold, "
                "or hyperparameter selection."
            ),
            "outer_holdout_manifest_path": str(OUTER_MANIFEST_PATH),
            "inner_manifest_path": str(INNER_MANIFEST_PATH),
            "static_feature_cache_path": str(STATIC_FEATURE_PATH),
        },
    )

    print("FINAL OUTER-HOLDOUT METRICS")
    print("=" * 72)
    for metric_name, metric_value in (
        exp1_final_holdout_results["metrics"].items()
    ):
        print(f"{metric_name:>32}: {metric_value}")
else:
    print(
        "Final holdout remains protected. Run candidate experiments on "
        "development first, then record one selected model."
    )
